### Importing Libraries

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import pandas as pd
import pickle
from config import ROOT_DIR
from src.models.baseline_model import create_baseline_recommender,create_recency_baseline
from src.models.als_model import train_als_model, create_als_recommender
from src.evaluation.metrics import evaluate_all, evaluate_candidates
from src.candidate_retrieval.PreviousPurchases import PreviousPurchases
from src.candidate_retrieval.ItemSimilarity import ItemSimilarityCandidates
from src.candidate_retrieval.RecencyCandidates import RecencyCandidates
from src.candidate_retrieval.AlsCandidates import ALSCandidates
from src.candidate_retrieval.CandidateRetrievalPipeline import CandidateGeneratorEnsemble

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [76]:
cleaned_df_folder = "input/cleaned"
def load_from_parquet(cleaned_df_name):
    print("Loading data from Parquet...")
    return pd.read_parquet(cleaned_df_folder + "/" + cleaned_df_name + ".parquet")

In [77]:
cols_needed = ['t_dat', 'customer_id', 'article_id','week']

train_transactions = pd.read_parquet(
    "input/cleaned/train_transactions.parquet", 
    columns=cols_needed
)

In [89]:

val_transactions = load_from_parquet("val_transactions")
articles = load_from_parquet("articles")

Loading data from Parquet...
Loading data from Parquet...


### Baseline model

In [ ]:
# Initialize the baseline recommender to get the top-k recommendations.
my_baseline_model = create_baseline_recommender(train_transactions)

# Evaluate the baseline model on the validation transactions.
evaluate_all(my_baseline_model, val_transactions)

Generating predictions for 68,984 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.0028985916189694194,
 'HitRate@12': 0.021120839614983185,
 'NDCG@12': 0.0053114661238347575,
 'Recall@12': 0.007748996773688122}

In [82]:
# Evaluate the recency baseline model on the validation transactions.
recency_model = create_recency_baseline(train_transactions)
evaluate_all(recency_model, val_transactions)


Generating predictions for 68,984 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.006313156988173997,
 'HitRate@12': 0.0663777107735127,
 'NDCG@12': 0.014016363034196056,
 'Recall@12': 0.026719109222937222}

### Collaborative filtering using ALS modelling

The training dataset is massive with around 1.3M unique customers and around 100k unique articles. This makes the collaborative filtering model to run very slowly on the entire dataset as the sparse matrix is massive.

So, down sampling only the last 3 months records would help in running the ALS model faster and also the data would be more relative to recommend as the customer's fashion sense from last 2 years would not matter much.

In [84]:
train_transactions["week"].min()

1

In [83]:
# Subset the training transactions to only include the last 60 days of data. 
train_transaction_subset = train_transactions[train_transactions['week'] <= 9]

In [53]:
print(len(train_transactions.customer_id.unique()), "unique users in the original training set.")
print(len(train_transaction_subset.customer_id.unique()), "unique users in the last 60 days of the training set.")

1356709 unique users in the original training set.
403116 unique users in the last 60 days of the training set.


In [62]:
als_model, user_map, item_map, reverse_item_map, item_user_matrix = train_als_model(
    train_transaction_subset,
    factors=100,
    iterations=15,
    regularization=0.01,
    alpha=40.0,
    random_seed=42
)

als_recommender = create_als_recommender(
    als_model=als_model,
    user_map=user_map,
    item_map=item_map,
    reverse_item_map=reverse_item_map,
    item_user_matrix=item_user_matrix,
    baseline=recency_model
)

1. User map shape: 403116
2. Item map shape: 36899
3. Reverse item map shape: 36899
Training Implicit ALS model...


  0%|          | 0/15 [00:00<?, ?it/s]

Training complete!


In [ ]:
als_file_path = ROOT_DIR / 'saved_models' / 'als_recommender_model.pkl'

In [ ]:
# Save the ALS model and mappings to a file for later use.
model_artifact = {
    'model': als_model,                
    'user_map': user_map,              
    'item_map': item_map,              
    'reverse_item_map': reverse_item_map,
    'item_user_matrix': item_user_matrix, 
}

with open(als_file_path, 'wb') as f:
    pickle.dump(model_artifact, f)

print("ALS model successfully saved!")

ALS model successfully saved!


In [69]:
# Load the ALS model and mappings from the file for evaluation.
with open(als_file_path, "rb") as f:
    artifacts = pickle.load(f)
    
als_recommender = create_als_recommender(
    als_model=artifacts["model"],
    user_map=artifacts["user_map"],
    item_map=artifacts["item_map"],
    reverse_item_map=artifacts["reverse_item_map"],
    item_user_matrix=artifacts["item_user_matrix"],
    baseline=recency_model  # Fallback for cold-start or unknown customers
)

In [74]:
evaluate_all(als_recommender, val_transactions)

Generating predictions for 68,984 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.023023678756249187,
 'HitRate@12': 0.1069668328887858,
 'NDCG@12': 0.03580656426402332,
 'Recall@12': 0.054490386669716666}

### Candidate Retrieval


In [ ]:
val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12}),
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
recs_df = ensemble.generate(users=val_customers, k=12)

print(f"Generated {len(recs_df):,} total candidate rows.")

Generating candidates via ensemble for 68,984 validation customers...
Generated 346,208 total candidate rows.


In [118]:

val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 20}),
    "item_cooccurrence": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 30
        },
    ),
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(recs_df):,} total candidate rows.")

Generating candidates via ensemble for 68,984 validation customers...
Generated candidate scores for users
Generated 1,474,109 total candidate rows.


In [116]:
evaluate_candidates(recs_df, val_transactions, k=50)

Preparing evaluation matrices...
Recall@50: 0.0668


0.06678938681997981

In [128]:
# 2. Count how many candidates each customer received
counts_per_user = recs_df.groupby('customer_id').size()

# 3. Group by candidate count to see the distribution
distribution = counts_per_user.value_counts().reset_index()
distribution.columns = ['num_candidates', 'num_customers']
distribution = distribution.sort_values('num_candidates')

# Print the binned breakdown
print(distribution.to_string(index=False))

# Also check how many validation customers got 0 candidates entirely
total_val_customers = len(val_customers)
customers_with_recs = counts_per_user.shape[0]
print(f"\nTotal Validation Customers: {total_val_customers:,}")
print(f"Customers with at least 1 candidate: {customers_with_recs:,}")
print(f"Customers with 0 candidates (Dropped/Cold Start): {total_val_customers - customers_with_recs:,}")

 num_candidates  num_customers
             20          19789
             21           2135
             22           2151
             23           1764
             24           1303
             25           1075
             26            769
             27            635
             28            484
             29            425
             30            320
             31            246
             32            194
             33            150
             34            143
             35            116
             36             88
             37             81
             38             60
             39             55
             40            261
             41             47
             42            101
             43            127
             44            293
             45            559
             46            872
             47           1147
             48           1568
             49           2116
             50           2572
        

In [127]:
from src.candidate_retrieval.RecencyCandidates import RecencyCandidates
val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 20}),
    "item_cooccurrence": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 30
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(recs_df):,} total candidate rows.")
evaluate_candidates(recs_df, val_transactions, k=70)

Generating candidates via ensemble for 68,984 validation customers...
Generated candidate scores for users
Generated 2,716,969 total candidate rows.
Preparing evaluation matrices...
Recall@70: 0.0998


0.09979706694395264

In [130]:
from src.candidate_retrieval.AlsCandidates import ALSCandidates
val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 15}),
    "item_cooccurrence": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
    "als": (ALSCandidates, {"lookback_weeks": 12, "k": 40})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(recs_df):,} total candidate rows.")
evaluate_candidates(recs_df, val_transactions, k=100)

1. User map shape: 467943
2. Item map shape: 39828
3. Reverse item map shape: 39828
Training Implicit ALS model...


  0%|          | 0/15 [00:00<?, ?it/s]

Training complete!
Generating candidates via ensemble for 68,984 validation customers...
Generated candidate scores for users


TypeError: 'NoneType' object is not callable